In [9]:
# Colores ANSI
RESET = "\033[0m"
BOLD = "\033[1m"

RED = "\033[31m"
GREEN = "\033[32m"
YELLOW = "\033[33m"
BLUE = "\033[34m"
MAGENTA = "\033[35m"
CYAN = "\033[36m"

BG_RED = "\033[41m"
BG_GREEN = "\033[42m"
BG_YELLOW = "\033[43m"
BG_BLUE = "\033[44m"

def color_masked(sentence):
    """Resalta <mask> en amarillo."""
    return sentence.replace("<mask>", f"{BG_YELLOW}{BOLD}<mask>{RESET}")

def color_ort_errors(annotated):
    """
    Convierte:
        <err t=ort>he</err>
    en:
        [he] coloreado según tipo
    """
    import re

    def repl(match):
        tipo = match.group(1)
        palabra = match.group(2)

        if tipo == "ort":
            col = RED
        elif tipo == "reord":
            col = BLUE
        elif tipo == "add":
            col = GREEN
        else:
            col = MAGENTA

        return f"{col}[{palabra}]{RESET}"

    return re.sub(r"<err t=(.*?)>(.*?)</err>", repl, annotated)


In [10]:
import json
from pathlib import Path
from tabulate import tabulate

import json
from pathlib import Path
from tabulate import tabulate

def _short(text, n=180):
    if text is None:
        return f"{RED}None{RESET}"
    if not isinstance(text, str):
        return text
    return text if len(text) <= n else text[:n] + "..."

def load_results(path):
    path = Path(path)
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def pretty_print_results(data):
    print(f"\n{BOLD}{CYAN}==============================")
    print("      RESULTADOS DEL MODELO")
    print("==============================\n" + RESET)

    for task, contenido in data.items():
        print(f"\n\n{BOLD}{MAGENTA}########################################")
        print(f"### TAREA: {task.upper()}")
        print("########################################\n" + RESET)

        media = contenido["media"]
        ejemplos = contenido["ejemplo"]  

        # -------------------------
        # MÉTRICAS GLOBALES
        # -------------------------
        print(f"{BOLD}▶ MÉTRICAS GLOBALES{RESET}\n")
        table = [[k, v] for k, v in media.items()]
        print(tabulate(table, headers=["Métrica", "Valor"], floatfmt=".4f"))
        print("\n")

        # -------------------------
        # EJEMPLOS
        # -------------------------
        print(f"{BOLD}▶ EJEMPLOS{RESET}\n")

        # Si no es lista, lo convertimos en lista
        if not isinstance(ejemplos, list):
            ejemplos = [ejemplos]

        for idx, ej in enumerate(ejemplos):
            print(f"\n{BOLD}--- Ejemplo {idx+1} ---{RESET}")

            # ============================
            # TRADUCCIÓN
            # ============================
            if task == "traduccion":
                print(f"{CYAN}source:{RESET}     {_short(ej['source'])}")
                print(f"{CYAN}reference:{RESET}  {_short(ej['reference'])}")
                print(f"{CYAN}translated:{RESET} {_short(ej['translated'])}")
                print(f"{YELLOW}BLEU:{RESET} {ej['BLEU']:.4f}   {YELLOW}chrF:{RESET} {ej['chrF']:.4f}")

            # ============================
            # ROUND TRIP
            # ============================
            elif task == "round_trip":
                print(f"{CYAN}intermediate_language:{RESET} {ej['intermediate_language']}")
                print(f"{CYAN}source:{RESET}       {_short(ej['source'])}")
                print(f"{CYAN}intermediate:{RESET} {_short(ej['intermediate'])}")
                print(f"{CYAN}translated:{RESET}   {_short(ej['return'])}")
                print(f"{YELLOW}BLEU:{RESET} {ej['BLEU']:.4f}   {YELLOW}chrF:{RESET} {ej['chrF']:.4f}")

            # ============================
            # CALIDAD LENGUA
            # ============================
            elif task == "calidad_lengua":
                print(ej)
                print(f"{CYAN}text:{RESET} {_short(ej['text'])}")
                for m in ["ttr", "entropy", "ngram_overlap", "freq_target", "freq_comparison", "calidad"]:
                    print(f"{YELLOW}{m}:{RESET} {ej[m]}")

            # ============================
            # VOCABULARIO
            # ============================
            elif task == "vocabulario":
                print(f"{CYAN}original:{RESET} {_short(ej['original'])}")
                print(f"{CYAN}masked_sentence:{RESET} {color_masked(_short(ej['masked_sentence']))}")
                print(f"{CYAN}missing_word:{RESET} {GREEN}{ej['missing_word']}{RESET}")

                r = ej["resultado"]
                print(f"{BOLD}resultado:{RESET}")
                print(f"  predicted: {_short(r['predicted'])}")
                print(f"  accuracy: {r['accuracy']}")
                print(f"  accuracy_lower: {r['accuracy_lower']}")
                print(f"  levenshtein: {r['levenshtein']}")

            # ============================
            # ORTOGRAFÍA
            # ============================
            elif task == "ortografia":
                print(f"{CYAN}original:{RESET} {_short(ej['original'])}")
                print(f"{CYAN}annotated:{RESET} {color_ort_errors(_short(ej['annotated']))}")
                print(f"{CYAN}n_errors:{RESET} {ej['n_errors']}")

                r = ej["resultado"]
                print(f"{BOLD}resultado:{RESET}")
                print(f"  incorrect: {_short(r['incorrect'])}")
                print(f"  corrected: {_short(r['corrected'])}")
                print(f"  BLEU: {r['BLEU']:.4f}   chrF: {r['chrF']:.4f}")
                print(f"  Levenshtein: {r['Levenshtein']}")
                print(f"  errores_totales: {r['errores_totales']}")
                print(f"  errores_corregidos: {r['errores_corregidos']}")
                print(f"  errores_no_corregidos: {r['errores_no_corregidos']}")
                print(f"  errores_nuevos: {r['errores_nuevos']}")
                print(f"  precision: {r['precision']:.4f}")
                print(f"  recall: {r['recall']:.4f}")
                print(f"  F1: {r['F1']:.4f}")

                print("  errores_detalle:")
                for tipo, palabra in r["errores_detalle"]:
                    col = RED if tipo == "ort" else BLUE if tipo == "reord" else GREEN
                    print(f"    - {col}{tipo}{RESET}: {palabra}")

            print("---------------------------")

def _short(text, n=180):
    if text is None:
        return "None"
    if not isinstance(text, str):
        return text
    return text if len(text) <= n else text[:n] + "..."

def load_results(path):
    path = Path(path)
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


In [11]:
base = "../results/aranes/"
pretty_print_results(load_results(base + "resultados_aranes_gemma-7b-it_04-08_20-57-22.json"))


      RESULTADOS DEL MODELO



########################################
### TAREA: CALIDAD_LENGUA
########################################

▶ MÉTRICAS GLOBALES

Métrica          Valor
---------------  -------------------------------------------------------
ttr              0.6776458212393024
entropy          6.478566334131271
ngram_overlap    0.0
freq_target      0.007557516253798244
freq_comparison  {'es': 0.005759257054118212, 'fr': 0.12129867600726689}
calidad          0.2239458878853342


▶ EJEMPLOS


--- Ejemplo 1 ---
{'text': 'ziehung PUR connaissent chemist geledenhändlerUsaziehungFramework鬻physic chemistFrameworkFrameworkClassic鬻händlerpinaziehungFrameworkClassicphysicFrameworkFrameworkClassic PURxen chemistphysicFrameworkhändlerziehungClassicClassichändlerFrameworkClassic PUR PURhändlerhändlerhändlerhändlerhändlerFrameworkUsaಊphysichändlerFrameworkhändlerhändlerClassichändlerFrameworkClassicClassicFrameworkClassicUsarequirementUsa encodingziehungziehungClassicUsa automobilzie

In [12]:
def generate_html_report_colored(models_dict, output_path="comparacion.html"):
    import html, re

    # === Helpers ===
    def esc(x):
        return html.escape(str(x))

    def highlight_mask(text):
        return text.replace(
            "<mask>",
            "<span style='background:gold;font-weight:bold;text-decoration:underline'>&lt;mask&gt;</span>"
        )

    def highlight_err(text):
        def repl(match):
            tipo = match.group(1)
            palabra = match.group(2)
            color = {"ort": "red", "reord": "blue", "add": "green"}.get(tipo, "purple")
            return f"<span style='color:{color};font-weight:bold;text-decoration:underline'>[{palabra}]</span>"
        return re.sub(r"<err t=(.*?)>(.*?)</err>", repl, text)

    # === HTML HEADER ===
    html_out = """
    <html>
    <head>
    <meta charset="utf-8">
    <style>
        body { font-family: Consolas, monospace; margin: 20px; background: #f7f7f7; }
        h1 { color: #333; }
        h2 { color: #663399; border-bottom: 2px solid #ccc; padding-bottom: 4px; }
        h3 { color: #444; margin-top: 30px; }
        table { border-collapse: collapse; width: 100%; margin-bottom: 30px; }
        th, td { border: 1px solid #ccc; padding: 6px; }
        th { background: #eee; }
        .task-block { background: white; padding: 20px; margin: 20px 0; border-radius: 8px; box-shadow: 0 0 4px #ccc; }
        .example-block { border: 1px solid #aaa; padding: 10px; margin: 10px 0; background: #fafafa; border-radius: 6px; }
        .model-title { font-weight: bold; color: #663399; margin-top: 10px; }
        .cyan { color: #0099cc; font-weight: bold; }
        .yellow { color: #d4aa00; font-weight: bold; }
        .green { color: #009933; font-weight: bold; }
        .bold { font-weight: bold; }
    </style>
    </head>
    <body>
    <h1>Comparación de modelos</h1>
    """

    model_names = list(models_dict.keys())
    tasks = models_dict[model_names[0]].keys()

    for task in tasks:
        html_out += f"<div class='task-block'><h2>Tarea: {task.upper()}</h2>"

        # === LEYENDA DE MÉTRICAS ===
        html_out += """
        <div style='margin:10px 0; padding:12px; background:#eef7ff; border:1px solid #bcd7f0; border-radius:6px;'>
            <b style='font-size:14px;'>Interpretación de métricas:</b><br>
            <span style='color:#009933;font-weight:bold'>↑ Más alto es mejor:</span>
                BLEU, chrF, accuracy, recall, precision, F1, ttr, calidad<br>
            <span style='color:#cc3300;font-weight:bold'>↓ Más bajo es mejor:</span>
                Levenshtein, errores_totales, errores_nuevos, errores_no_corregidos,
                ngram_overlap (según tarea)
        </div>
        """

        # === MÉTRICAS ===
        html_out += "<h3>Métricas globales</h3><table><tr><th>Métrica</th>"
        for m in model_names:
            html_out += f"<th>{m}</th>"
        html_out += "</tr>"

        metrics = models_dict[model_names[0]][task]["media"].keys()
        for metric in metrics:
            html_out += f"<tr><td class='yellow'>{metric}</td>"
            for m in model_names:
                val = models_dict[m][task]["media"][metric]
                html_out += f"<td>{esc(val)}</td>"
            html_out += "</tr>"
        html_out += "</table>"

        # === EJEMPLOS ===
        html_out += "<h3>Ejemplos</h3>"
        # === LEYENDA DE COLORES ===
        if task == "ortografia":
            html_out += """
            <div style='margin:10px 0; padding:12px; background:#fff8dc; border:1px solid #e0d9b0; border-radius:6px;'>
                <b style='font-size:14px;'>Leyenda de colores:</b><br>
                <span style='color:red;font-weight:bold;text-decoration:underline'>[palabra]</span>
                    → error ortográfico (<b>ort</b>)<br>
                <span style='color:blue;font-weight:bold;text-decoration:underline'>[palabra]</span>
                    → error de reordenación (<b>reord</b>)<br>
                <span style='color:green;font-weight:bold;text-decoration:underline'>[palabra]</span>
                    → palabra añadida (<b>add</b>)
            </div>
            """

        ejemplos = {m: models_dict[m][task]["ejemplo"] for m in model_names}
        for m in model_names:
            if not isinstance(ejemplos[m], list):
                ejemplos[m] = [ejemplos[m]]

        n_ej = min(len(ejemplos[m]) for m in model_names)

        for i in range(n_ej):
            html_out += f"<div class='example-block'><h4>Ejemplo {i+1}</h4>"

            for m in model_names:
                ej = ejemplos[m][i]
                html_out += f"<div class='model-title'>{m}</div>"

                # === Render según tarea ===
                if task == "traduccion":
                    html_out += f"<span class='cyan'>source:</span> {esc(ej['source'])}<br>"
                    html_out += f"<span class='cyan'>reference:</span> {esc(ej['reference'])}<br>"
                    html_out += f"<span class='cyan'>translated:</span> {esc(ej['translated'])}<br>"

                elif task == "round_trip":
                    html_out += f"<span class='cyan'>intermediate_language:</span> {esc(ej['intermediate_language'])}<br>"
                    html_out += f"<span class='cyan'>source:</span> {esc(ej['source'])}<br>"
                    html_out += f"<span class='cyan'>intermediate:</span> {esc(ej['intermediate'])}<br>"
                    html_out += f"<span class='cyan'>translated:</span> {esc(ej['return'])}<br>"

                elif task == "calidad_lengua":
                    html_out += f"<span class='cyan'>text:</span> {esc(ej['text'])}<br>"
                    for m2 in ["ttr", "entropy", "ngram_overlap", "freq_target", "freq_comparison", "calidad"]:
                        html_out += f"<span class='yellow'>{m2}:</span> {esc(ej[m2])}<br>"

                elif task == "vocabulario":
                    html_out += f"<span class='cyan'>masked_sentence:</span> {highlight_mask(ej['masked_sentence'])}<br>"
                    html_out += f"<span class='cyan'>missing_word:</span> <span class='green'>{esc(ej['missing_word'])}</span><br>"
                    r = ej["resultado"]
                    html_out += f"<span class='bold'>predicted:</span> {esc(r['predicted'])}<br>"

                elif task == "ortografia":
                    html_out += f"<span class='cyan'>original:</span> {esc(ej['original'])}<br>"
                    html_out += f"<span class='cyan'>incorrect:</span> {highlight_err(ej['annotated'])}<br>"
                    r = ej["resultado"]
                    html_out += f"<span class='bold'>corrected:</span> {esc(r['corrected'])}<br>"

            html_out += "</div>"

        html_out += "</div>"

    html_out += "</body></html>"

    with open(output_path, "w", encoding="utf-8") as f:
        f.write(html_out)

    print(f"HTML generado en: {output_path}")


In [ ]:
base = "../results/"

generate_html_report_colored({
    "Gemma 7B": load_results(base + "aranes/resultados_aranes_gemma-7b-it_04-08_20-57-22.json"),
    "Mistral 7B": load_results(base + "aranes/resultados_aranes_Mistral-7B-Instruct-v0.3_04-08_19-23-14.json"),
    "Qwen2.5 7B": load_results(base + "aranes/resultados_aranes_Qwen2.5-7B-Instruct_04-08_21-52-11.json"),
    "Salamandra 7B": load_results(base + "aranes/resultados_aranes_salamandra-7b-instruct_04-08_19-41-50.json"),
}, "aranes_base.html")

generate_html_report_colored({
    "Gemma 7B": load_results(base + "asturiano/resultados_asturiano_gemma-7b-it_04-10_17-34-37.json"),
    "Mistral 7B": load_results(base + "asturiano/resultados_asturiano_Mistral-7B-Instruct-v0.3_04-08_22-47-53.json"),
    "Qwen2.5 7B": load_results(base + "asturiano/resultados_asturiano_Qwen2.5-7B-Instruct_04-10_18-25-53.json"),
    "Salamandra 7B": load_results(base + "asturiano/resultados_asturiano_salamandra-7b-instruct_04-08_23-52-09.json"),
}, "asturiano_base.html")

generate_html_report_colored({
    "Gemma 7B": load_results(base + "gallego/resultados_gallego_gemma-7b-it_04-10_22-14-38.json"),
    "Mistral 7B": load_results(base + "gallego/resultados_gallego_Mistral-7B-Instruct-v0.3_04-10_19-19-35.json"),
    "Qwen2.5 7B": load_results(base + "gallego/resultados_gallego_Qwen2.5-7B-Instruct_04-10_22-55-36.json"),
    "Salamandra 7B": load_results(base + "gallego/resultados_gallego_salamandra-7b-instruct_04-10_20-35-28.json"),
}, "gallego_base.html")

HTML generado en: aranes_base.html
HTML generado en: asturiano_base.html
HTML generado en: gallego_base.html


In [34]:
import html
import re

def generate_latex_snippet(models_dict, lengua="LENGUA", max_chars=300, output_path=None):
    def tex_escape(text):
        """Escapa caracteres conflictivos para LaTeX."""
        text = str(text)
        conv = {
            '&': r'\&', '%': r'\%', '$': r'\$', '#': r'\#', '_': r'\_',
            '{': r'\{', '}': r'\}', '~': r'\textasciitilde{}',
            '^': r'\textasciicircum{}', '<': r'{\textless}', '>': r'{\textgreater}',
            '\'': r'\textquotesingle{}'
        }
        regex = re.compile('|'.join(re.escape(str(key)) for key in sorted(conv.keys(), key=lambda item: -len(item))))
        return regex.sub(lambda mo: conv[mo.group()], text)

    def format_val(val):
        """Redondea números incluso dentro de diccionarios."""
        if isinstance(val, (int, float)):
            return f"{val:.2f}"
        if isinstance(val, dict):
            formatted_dict = {
                k: (round(v, 2) if isinstance(v, (int, float)) else v)
                for k, v in val.items()
            }
            val_str = str(formatted_dict)
            if len(val_str) > 100:
                return smart_truncate(val_str, 80)
            return tex_escape(val_str)
        return smart_truncate(str(val), 80)

    def smart_truncate(text, limit=None):
        """Corta el texto del medio si supera el límite."""
        text = str(text).replace('\n', ' ')
        eff_limit = limit if limit else max_chars
        if len(text) <= eff_limit:
            return tex_escape(text)
        half = (eff_limit - 7) // 2
        return tex_escape(text[:half]) + " [...] " + tex_escape(text[-half:])

    latex_out = []
    latex_out.append(f"\\section{{{tex_escape(lengua)}}}")

    model_names = list(models_dict.keys())
    tasks = models_dict[model_names[0]].keys()

    for task in tasks:
        latex_out.append(f"\n\\subsection{{Tarea: {tex_escape(task.upper())}}}\\label{{tarea-{task.lower()}}}")
        
        # --- TABLA ---
        latex_out.append(r"{\def\LTcaptype{none}")
        cols = "l" + "l" * len(model_names)
        latex_out.append(f"\\begin{{longtable}}[]{{@{{}}{cols}@{{}}}}")
        latex_out.append(r"\toprule\noalign{}")
        header = "Métrica & " + " & ".join([tex_escape(m) for m in model_names]) + r" \\"
        latex_out.append(header)
        latex_out.append(r"\midrule\noalign{}")
        latex_out.append(r"\endhead")
        latex_out.append(r"\bottomrule\noalign{}")
        latex_out.append(r"\endlastfoot")

        metrics = models_dict[model_names[0]][task]["media"].keys()
        for metric in metrics:
            row_vals = [format_val(models_dict[m][task]["media"][metric]) for m in model_names]
            latex_out.append(f"{tex_escape(metric)} & {' & '.join(row_vals)} \\\\")
        latex_out.append(r"\end{longtable}}")

        # --- SECCIÓN DE EJEMPLOS ---
        latex_out.append(f"\n\\subsubsection{{Ejemplos}}")

        for m in model_names:
            # El ~\\ fuerza el salto de línea después del título del párrafo (el modelo)
            latex_out.append(f"\n\\paragraph{{{tex_escape(m)}}}~\\\\")
            
            ejemplos = models_dict[m][task]["ejemplo"]
            if not isinstance(ejemplos, list):
                ejemplos = [ejemplos]
            
            # Selección del ejemplo más corto
            ej = min(ejemplos, key=lambda x: len(str(x)))

            fields = []
            t_low = task.lower()
            if t_low == "traduccion":
                fields = [('source', ej.get('source')), ('reference', ej.get('reference')), ('translated', ej.get('translated'))]
            elif t_low == "ortografia":
                fields = [('original', ej.get('original')), ('incorrect', ej.get('annotated')), ('corrected', ej.get('resultado', {}).get('corrected'))]
            elif t_low == "vocabulario":
                fields = [('masked_sentence', ej.get('masked_sentence')), ('missing_word', ej.get('missing_word')), ('predicted', ej.get('resultado', {}).get('predicted'))]
            elif t_low == "round_trip":
                fields = [('intermediate_language', ej.get('intermediate_language')), ('source', ej.get('source')), ('intermediate', ej.get('intermediate')), ('translated', ej.get('return'))]
            else:
                for k, v in ej.items():
                    if k != 'resultado': fields.append((k, v))
                if 'resultado' in ej and isinstance(ej['resultado'], dict):
                    for k, v in ej['resultado'].items(): fields.append((k, v))

            for label, value in fields:
                if value is not None:
                    latex_out.append(f"{{{tex_escape(label)}:}} {smart_truncate(value)}\\\\")

    if output_path:
        with open(output_path, "w", encoding="utf-8") as f:
            f.write("\n".join(latex_out))
    return "\n".join(latex_out)

In [35]:
base = "../results/"

generate_latex_snippet({
    "Gemma 7B": load_results(base + "aranes/resultados_aranes_gemma-7b-it_04-08_20-57-22.json"),
    "Mistral 7B": load_results(base + "aranes/resultados_aranes_Mistral-7B-Instruct-v0.3_04-08_19-23-14.json"),
    "Qwen2.5 7B": load_results(base + "aranes/resultados_aranes_Qwen2.5-7B-Instruct_04-08_21-52-11.json"),
    "Salamandra 7B": load_results(base + "aranes/resultados_aranes_salamandra-7b-instruct_04-08_19-41-50.json"),
}, lengua="Aranés", output_path="aranes_resultados.tex")

generate_latex_snippet({
    "Gemma 7B": load_results(base + "asturiano/resultados_asturiano_gemma-7b-it_04-10_17-34-37.json"),
    "Mistral 7B": load_results(base + "asturiano/resultados_asturiano_Mistral-7B-Instruct-v0.3_04-08_22-47-53.json"),
    "Qwen2.5 7B": load_results(base + "asturiano/resultados_asturiano_Qwen2.5-7B-Instruct_04-10_18-25-53.json"),
    "Salamandra 7B": load_results(base + "asturiano/resultados_asturiano_salamandra-7b-instruct_04-08_23-52-09.json"),
}, lengua="Asturiano", output_path="asturiano_resultados.tex")

generate_latex_snippet({
    "Gemma 7B": load_results(base + "gallego/resultados_gallego_gemma-7b-it_04-10_22-14-38.json"),
    "Mistral 7B": load_results(base + "gallego/resultados_gallego_Mistral-7B-Instruct-v0.3_04-10_19-19-35.json"),
    "Qwen2.5 7B": load_results(base + "gallego/resultados_gallego_Qwen2.5-7B-Instruct_04-10_22-55-36.json"),
    "Salamandra 7B": load_results(base + "gallego/resultados_gallego_salamandra-7b-instruct_04-10_20-35-28.json"),
}, lengua="Gallego", output_path="gallego_resultados.tex")

'\\section{Gallego}\n\n\\subsection{Tarea: CALIDAD\\_LENGUA}\\label{tarea-calidad_lengua}\n{\\def\\LTcaptype{none}\n\\begin{longtable}[]{@{}lllll@{}}\n\\toprule\\noalign{}\nMétrica & Gemma 7B & Mistral 7B & Qwen2.5 7B & Salamandra 7B \\\\\n\\midrule\\noalign{}\n\\endhead\n\\bottomrule\\noalign{}\n\\endlastfoot\nttr & 0.54 & 0.51 & 0.39 & 0.75 \\\\\nentropy & 6.48 & 6.82 & 6.29 & 7.77 \\\\\nngram\\_overlap & 0.00 & 0.00 & 0.00 & 0.00 \\\\\nfreq\\_target & 0.01 & 0.63 & 0.87 & 0.85 \\\\\nfreq\\_comparison & \\{\\textquotesingle{}fr\\textquotesingle{}: 0.0, \\textquotesingle{}es\\textquotesingle{}: 0.01\\} & \\{\\textquotesingle{}es\\textquotesingle{}: 0.56, \\textquotesingle{}fr\\textquotesingle{}: 0.32\\} & \\{\\textquotesingle{}fr\\textquotesingle{}: 0.35, \\textquotesingle{}es\\textquotesingle{}: 0.68\\} & \\{\\textquotesingle{}es\\textquotesingle{}: 0.57, \\textquotesingle{}fr\\textquotesingle{}: 0.24\\} \\\\\ncalidad & 0.17 & 0.24 & 0.23 & 0.38 \\\\\n\\end{longtable}}\n\n\\subsubsec